In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [2]:
ds = pd.read_csv("cleaned_data.csv")

In [3]:
#Lets divide in independent(X) and dependent(y)
X = ds[["company", "name", "year", "kms_driven", "fuel_type"]]
y = ds[["Price"]]

In [4]:
len(X["name"].unique())

444

In [4]:
#Lets make column transformer who will help us with OHE
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder()
ohe.fit(X[["company", "name", "fuel_type"]])

OneHotEncoder()

In [5]:
#Make column transformer
from sklearn.compose import make_column_transformer
ct = make_column_transformer((OneHotEncoder(categories = ohe.categories_), 
                              ["company", "name", "fuel_type"]), remainder='passthrough')

In [15]:
#Lets make Model object
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

reg = LinearRegression()
regDec = DecisionTreeRegressor()
regRFR = RandomForestRegressor(n_estimators = 10)

# Model initialization
regXGB = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

In [16]:
#Lets make pipeline whose first part is CT and second is model
from sklearn.pipeline import make_pipeline
pipe = make_pipeline(ct, reg)
pipeDec = make_pipeline(ct, regDec)
pipeRFR = make_pipeline(ct, regRFR)
pipeXGB = make_pipeline(ct, regXGB)

In [18]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
scores = []
scoresDec = []
scoresRFR = []
scoresXGB = []
for i in range(0, 100):
#Lets divide data into training and test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.1, random_state = i)
    #Lets train model(ct->reg)-pipe
    pipe.fit(X_train, y_train)
    pipeDec.fit(X_train, y_train)
    pipeRFR.fit(X_train, y_train)
    pipeXGB.fit(X_train, y_train)

    #Prediction and Score calculation - Linear Regression
    y_pred = pipe.predict(X_test)
    y_pred = pd.DataFrame(data = y_pred, columns = ["Result"])
    final = pd.concat([y_test.reset_index(drop = True), y_pred.reset_index(drop = True)], axis = 1)
    score = r2_score(final["Result"], final["Price"])
    scores.append(score)

     #Prediction and Score calculation - Decision Tree Regression
    y_pred = pipeDec.predict(X_test)
    y_pred = pd.DataFrame(data = y_pred, columns = ["Result"])
    final = pd.concat([y_test.reset_index(drop = True), y_pred.reset_index(drop = True)], axis = 1)
    score = r2_score(final["Result"], final["Price"])
    scoresDec.append(score)

    #Prediction and Score calculation - Random Forest Regression
    y_pred = pipeRFR.predict(X_test)
    y_pred = pd.DataFrame(data = y_pred, columns = ["Result"])
    final = pd.concat([y_test.reset_index(drop = True), y_pred.reset_index(drop = True)], axis = 1)
    score = r2_score(final["Result"], final["Price"])
    scoresRFR.append(score)

    #Prediction and Score calculation - XGB Regression
    y_pred = pipeXGB.predict(X_test)
    y_pred = pd.DataFrame(data = y_pred, columns = ["Result"])
    final = pd.concat([y_test.reset_index(drop = True), y_pred.reset_index(drop = True)], axis = 1)
    score = r2_score(final["Result"], final["Price"])
    scoresXGB.append(score)

In [19]:
index = np.argmax(scores)
indexDec = np.argmax(scoresDec)
indexRFR = np.argmax(scoresRFR)
indexXGB = np.argmax(scoresXGB)
print("LR", index, scores[index])
print("DTR", indexDec, scoresDec[indexDec])
print("RFR", indexRFR, scoresDec[indexRFR])
print("XGB", indexRFR, scoresDec[indexXGB])

LR 7 0.7937125816216408
DTR 62 0.6743954840386464
RFR 40 0.6529674716932543
XGB 40 0.23801113863336731


In [11]:
#Lets divide data into training and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.1, random_state = index)
#Lets train model(ct->reg)-pipe
pipe.fit(X_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('onehotencoder',
                                                  OneHotEncoder(categories=[array(['Audi', 'BMW', 'Chevrolet', 'Datsun', 'Fiat', 'Force', 'Ford',
       'Hindustan', 'Honda', 'Hyundai', 'Jaguar', 'Jeep', 'Land',
       'Mahindra', 'Maruti', 'Mercedes', 'Mini', 'Mitsubishi', 'Nissan',
       'Renault', 'Skoda', 'Tata', 'Toyota', 'Vol...
       'i10 Magna 1.2', 'i10 Magna 1.2 Kappa2', 'i10 Sportz',
       'i10 Sportz 1.2', 'i20', 'i20 Active', 'i20 Active 1.2 SX',
       'i20 Active 1.4L SX O', 'i20 Asta', 'i20 Asta 1.2',
       'i20 Asta 1.4 CRDI 6 Speed', 'i20 Magna', 'i20 Magna 1.2',
       'i20 Magna O 1.2', 'i20 Select Variant', 'i20 Sportz 1.2',
       'i20 Sportz 1.4 CRDI'], dtype=object),
                                                                            array(['Diesel', 'LPG', 'Petrol'], dtype=object)]),
                                                  ['company', 'name',
                                                   'fuel_type'])])),
                ('linearregression', LinearRegression())])

In [12]:
data = [["Ford", "Figo", 2018, 70000, "Petrol"]]
columns = ["company", "name", "year", "kms_driven", "fuel_type"]
myinput = pd.DataFrame(data = data, columns = columns)
result = pipe.predict(myinput)
round(result[0,0])

374787

In [13]:
#Lets export pipe for future use in API/Web Application
import pickle as pkl
pkl.dump(pipe, open("CPP.pkl", "wb+"))